In [ ]:
!pip install rasterio geopandas shapely scikit-image "transformers>=4.56" -q

In [ ]:
import os, gc, cv2, time, json, contextlib

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

import rasterio
import rasterio.features
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import shape, Polygon, MultiPolygon

from skimage.measure import label as sk_label
from skimage.segmentation import watershed, relabel_sequential

from google.colab import userdata

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# HUGGING FACE TOKEN (Colab secret HF_TOKEN) — needed for the gated DINOv3
# weights. Without it (or before the licence is accepted on the model page)
# the notebook falls back to DINOv2 automatically.
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print('HF token found' if HF_TOKEN else 'No HF token → DINOv3 unavailable, DINOv2 will be used')

## MODEL ARCHITECTURE

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def amp_dtype(device):
    """bf16 on GPUs that support it (A100, L4…), fp16 otherwise (T4), None on CPU."""
    if device.type != 'cuda':
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def autocast_ctx(device):
    dtype = amp_dtype(device)
    return torch.autocast('cuda', dtype=dtype) if dtype else contextlib.nullcontext()


class GatedFusion(nn.Module):
    """
    Spatially aligned fusion of foundation-model features into a U-Net level.

        e   = refine(proj(foundation_feat))       resized to the level's grid
        g   = sigmoid(conv1x1([x, e]))            per-pixel, per-channel gate
        out = x + gamma * g * e

    Why this replaces the old cross-attention block:
      • The foundation features are already on the same spatial grid as the
        U-Net features, so each position simply reads its own location. The
        old attention had no positional encoding, so every position attended
        to an unordered bag of SAM tokens and lost where things were.
      • gamma starts at 0.5 (not 0) and trains at the normal learning rate
        without weight decay, so the fusion can actually open. The old gate
        was capped at ~0.03 by its 1e-5 LR and ended at 0.012.
    """
    def __init__(self, ch: int, ext_ch: int, init_gamma: float = 0.5):
        super().__init__()
        self.proj   = nn.Sequential(nn.Conv2d(ext_ch, ch, 1, bias=False),
                                    nn.BatchNorm2d(ch), nn.GELU())
        self.refine = nn.Sequential(nn.Conv2d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm2d(ch))
        self.gate   = nn.Sequential(nn.Conv2d(2 * ch, ch, 1), nn.Sigmoid())
        self.gamma  = nn.Parameter(torch.tensor(float(init_gamma)))

    def forward(self, x: torch.Tensor, ext: torch.Tensor) -> torch.Tensor:
        e = self.proj(ext.to(x.dtype))
        if e.shape[-2:] != x.shape[-2:]:
            e = F.interpolate(e, size=x.shape[-2:], mode='bilinear', align_corners=False)
        e = self.refine(e)
        g = self.gate(torch.cat([x, e], dim=1))
        return x + self.gamma * g * e


# ── FracTAL ResUNet components (unchanged) ───────────────────────────────────

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, max(channels//reduction, 4)), nn.ReLU(),
            nn.Linear(max(channels//reduction, 4), channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 1)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(x))


class FracTALUnit(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ch_att = ChannelAttention(channels)
        self.sp_att = SpatialAttention(channels)
        self.alpha  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        a = torch.clamp(self.alpha, 0.0, 1.0)
        return a * self.ch_att(x) + (1-a) * self.sp_att(x)


class FracTALResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
        )
        self.fractal = FracTALUnit(out_ch)
        self.proj    = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.proj(x) + self.fractal(self.conv(x))


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks=2, dilation=1):
        super().__init__()
        blocks = [FracTALResBlock(in_ch, out_ch, dilation)]
        for _ in range(n_blocks-1):
            blocks.append(FracTALResBlock(out_ch, out_ch, dilation))
        self.blocks = nn.Sequential(*blocks)
        self.pool   = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.blocks(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.block = FracTALResBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))


# ── FracTAL ResUNet + frozen foundation encoder ──────────────────────────────

class FoundationFracTALResUNet(nn.Module):
    """
    FracTAL ResUNet with a frozen foundation encoder fused at three levels:

        level   resolution (512 input)   U-Net features fused
        's3'    64×64                    enc3 skip  (fine structure, edges)
        's4'    32×32                    enc4 skip
        'bott'  32×32                    bottleneck (field extent, semantics)

    `foundation` is any module returning a list of feature maps and exposing
    `out_channels` (one entry per map). `fusion_sources` says which of those
    maps feeds each level, e.g. {'s3': 0, 's4': 1, 'bott': 2}.
    With foundation=None the model is the plain FracTAL ResUNet baseline.
    """
    def __init__(self, foundation=None, fusion_sources=None,
                 num_classes: int = 2, base_ch: int = 32):
        super().__init__()
        c = base_ch
        self.foundation = foundation
        self.fusion_sources = dict(fusion_sources or {})

        self.enc1       = EncoderBlock(3,   c,   n_blocks=2)
        self.enc2       = EncoderBlock(c,   c*2, n_blocks=2)
        self.enc3       = EncoderBlock(c*2, c*4, n_blocks=3)
        self.enc4       = EncoderBlock(c*4, c*8, n_blocks=3)
        self.bottleneck = nn.Sequential(
            FracTALResBlock(c*8,  c*16, dilation=2),
            FracTALResBlock(c*16, c*16, dilation=4),
            FracTALResBlock(c*16, c*16, dilation=2),
        )

        level_ch = {'s3': c*4, 's4': c*8, 'bott': c*16}
        self.fusions = nn.ModuleDict()
        if foundation is not None:
            for level, src in self.fusion_sources.items():
                self.fusions[level] = GatedFusion(level_ch[level], foundation.out_channels[src])

        self.dec4 = DecoderBlock(c*16, c*8, c*8)
        self.dec3 = DecoderBlock(c*8,  c*4, c*4)
        self.dec2 = DecoderBlock(c*4,  c*2, c*2)
        self.dec1 = DecoderBlock(c*2,  c,   c)

        self.seg_head   = nn.Conv2d(c, num_classes, 1)
        self.bound_head = nn.Conv2d(c, 1, 1)
        self.dist_head  = nn.Conv2d(c, 1, 1)

    def _fuse(self, level, x, feats):
        if level in self.fusions:
            return self.fusions[level](x, feats[self.fusion_sources[level]])
        return x

    def forward(self, x: torch.Tensor) -> dict:
        feats = self.foundation(x) if self.foundation is not None else None

        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x     = self.bottleneck(x)

        s3 = self._fuse('s3',   s3, feats)
        s4 = self._fuse('s4',   s4, feats)
        x  = self._fuse('bott', x,  feats)

        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)

        return {'seg':   self.seg_head(x),
                'bound': self.bound_head(x),
                'dist':  self.dist_head(x)}

    def train(self, mode: bool = True):
        super().train(mode)
        if self.foundation is not None:
            self.foundation.eval()   # frozen: keep its norm/dropout layers in eval mode
        return self

    def fusion_strengths(self) -> dict:
        return {k: float(f.gamma.detach()) for k, f in self.fusions.items()}

    def trainable_state_dict(self) -> dict:
        """Everything except the frozen foundation weights (loaded separately)."""
        return {k: v for k, v in self.state_dict().items()
                if not k.startswith('foundation.')}

    def load_trainable_state_dict(self, state_dict):
        missing, unexpected = self.load_state_dict(state_dict, strict=False)
        missing = [k for k in missing if not k.startswith('foundation.')]
        if missing or unexpected:
            raise RuntimeError(f'Checkpoint mismatch. missing={missing[:5]} unexpected={unexpected[:5]}')
        return self


# Backbones tried in order; the first one that loads is used.
#   DINOv3 weights are free to use under Meta's DINOv3 License but GATED on
#   Hugging Face: accept the licence on the model page once, then add an
#   HF_TOKEN secret in Colab. If that isn't done, loading falls back to DINOv2
#   (Apache-2.0, not gated).
DINO_CANDIDATES = [
    'facebook/dinov3-vitl16-pretrain-sat493m',    # DINOv3 ViT-L/16, satellite-pretrained (SAT-493M)
    'facebook/dinov3-vitb16-pretrain-lvd1689m',   # DINOv3 ViT-B/16, web-image pretrained
    'facebook/dinov2-with-registers-base',        # DINOv2 ViT-B/14 + registers (fallback)
]

# Normalisation published with the DINOv3 satellite models; everything else
# uses ImageNet statistics.
SAT_MEAN = (0.430, 0.411, 0.296)
SAT_STD  = (0.213, 0.156, 0.143)


class DINOEncoder(nn.Module):
    """
    Frozen DINOv3 / DINOv2 ViT returning patch-token maps from three depths.

    Input : (B,3,H,W) ImageNet-normalised, H and W multiples of 16
    Output: [shallow, middle, deep] each (B, C, H/16, W/16)

    The tile is resized so the token grid is exactly H/16 × W/16 whatever the
    patch size (16 for DINOv3, 14 for DINOv2), keeping it aligned with the
    U-Net's 1/16 bottleneck. Three depths feed three U-Net levels: shallower
    layers carry more local/edge detail, the last layer the most semantics.
    """
    def __init__(self, model_name: str, layer_fractions=(0.5, 0.75, 1.0), token=None):
        super().__init__()
        from transformers import AutoModel
        self.model_name = model_name
        self.backbone   = AutoModel.from_pretrained(model_name, token=token)
        self.backbone.requires_grad_(False)
        self.backbone.eval()

        cfg = self.backbone.config
        self.patch   = cfg.patch_size
        n_layers     = cfg.num_hidden_layers
        # hidden_states[0] is the patch embedding; [i] is the output of block i
        self.layers  = [max(1, round(f * n_layers)) for f in layer_fractions]
        self.out_channels = [cfg.hidden_size] * len(self.layers)

        mean, std = (SAT_MEAN, SAT_STD) if 'sat493m' in model_name else (IMAGENET_MEAN, IMAGENET_STD)
        self.register_buffer('mean',    torch.tensor(mean).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('std',     torch.tensor(std).view(1, 3, 1, 1),  persistent=False)
        self.register_buffer('in_mean', torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('in_std',  torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),  persistent=False)

    @torch.no_grad()
    def forward(self, x: torch.Tensor):
        B, _, H, W = x.shape
        gh, gw = H // 16, W // 16
        x = (x * self.in_std + self.in_mean - self.mean) / self.std
        size = (gh * self.patch, gw * self.patch)
        if x.shape[-2:] != size:
            x = F.interpolate(x, size=size, mode='bilinear', align_corners=False)

        hs = self.backbone(pixel_values=x, output_hidden_states=True).hidden_states
        feats = []
        for i in self.layers:
            t = hs[i]
            t = t[:, t.shape[1] - gh * gw:, :]          # drop CLS + register tokens
            t = F.layer_norm(t, t.shape[-1:])
            feats.append(t.transpose(1, 2).reshape(B, -1, gh, gw))
        return feats


def load_dino_encoder(candidates=DINO_CANDIDATES, token=None):
    last_err = None
    for name in candidates:
        try:
            enc = DINOEncoder(name, token=token)
            print(f'✓ Loaded {name}  (patch={enc.patch}, layers={enc.layers}, '
                  f'dim={enc.out_channels[0]})')
            return enc
        except Exception as e:  # gated / not accepted / offline
            print(f'✗ {name}: {type(e).__name__}: {str(e).splitlines()[0][:120]}')
            last_err = e
    raise RuntimeError('No DINO backbone could be loaded') from last_err

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Loading Model (DINO)
# ════════════════════════════════════════════════════════════════════════════

MODEL_CHECKPOINT = '/content/drive/MyDrive/AGRI/CropField_Segmentation/result/dino-fractal-resunet-512-v01/best.pt'
OUTPUT_DIR       = '/content/drive/MyDrive/AGRI/CropField_Segmentation/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cpu':
    print('⚠ Running on CPU is slow (especially with ViT-L). Use a GPU runtime.')

ckpt = torch.load(MODEL_CHECKPOINT, map_location='cpu', weights_only=False)
MODEL_CONFIG = ckpt['model_config']
print('Model config:', MODEL_CONFIG)

# Load exactly the backbone the model was trained with (no fallback here)
encoder = load_dino_encoder([MODEL_CONFIG['backbone_name']], token=HF_TOKEN)

model = FoundationFracTALResUNet(
    foundation     = encoder,
    fusion_sources = MODEL_CONFIG['fusion_sources'],
    num_classes    = MODEL_CONFIG['num_classes'],
    base_ch        = MODEL_CONFIG['base_ch'],
)
model.load_trainable_state_dict(ckpt['state_dict'])
model = model.to(DEVICE).eval()
if DEVICE.type == 'cuda':
    model = model.to(memory_format=torch.channels_last)   # faster convolutions under AMP

print(f"Checkpoint epoch  : {ckpt['epoch']}  |  val FieldF1 {ckpt['val_metrics']['field_f1']:.4f}")
print(f'Fusion strengths  : {model.fusion_strengths()}')

In [ ]:
POSTPROCESS_DEFAULTS = {
    'extent_thresh' : 0.50,  # field probability above which a pixel belongs to a field
    'marker_dist'   : 0.40,  # distance-head value that marks a field's core
    'marker_bound'  : 0.30,  # ...only where the boundary head is below this
    'min_marker_px' : 20,    # ignore tiny seeds (noise)
    'min_field_px'  : 100,   # drop fields smaller than this
    'smooth_sigma'  : 1.0,   # Gaussian blur (px) of the maps first: straighter field edges
}


def delineate_fields(field_p: np.ndarray, bound_p: np.ndarray, dist_p: np.ndarray,
                     cfg: dict = POSTPROCESS_DEFAULTS) -> np.ndarray:
    """
    Probability maps → instance label map (0 = background, 1..N = fields).

    Seeded watershed, as in the original FracTAL ResUNet paper
    (after a light Gaussian blur of the three maps, see smooth_sigma):
      1. extent  = pixels the segmentation head calls field
      2. markers = field cores: high distance-to-edge, low boundary probability
      3. flood the energy (boundary − distance) from the markers inside the
         extent, so touching fields split along the predicted boundary while
         each keeps its full extent (no boundary strip is cut away, so no
         buffering is needed afterwards)
      4. extent pieces that received no marker are kept as their own fields
    """
    sigma = cfg.get('smooth_sigma', 0.0)
    if sigma > 0:   # removes pixel noise that would make the field edges wiggle
        blur = lambda a: cv2.GaussianBlur(a.astype(np.float32, copy=False), (0, 0), sigma)
        field_p, bound_p, dist_p = blur(field_p), blur(bound_p), blur(dist_p)

    extent  = field_p > cfg['extent_thresh']
    cores   = (dist_p > cfg['marker_dist']) & (bound_p < cfg['marker_bound']) & extent
    markers = _drop_small(sk_label(cores, connectivity=1), cfg['min_marker_px'])

    energy  = bound_p.astype(np.float32) - dist_p.astype(np.float32)
    labels  = watershed(energy, markers, mask=extent)

    orphans = sk_label(extent & (labels == 0), connectivity=1)
    if orphans.max() > 0:
        labels = np.where(orphans > 0, orphans + labels.max(), labels)

    return _drop_small(labels, cfg['min_field_px']).astype(np.int32)


def _drop_small(labels: np.ndarray, min_px: int) -> np.ndarray:
    """Remove labelled regions smaller than min_px and relabel 1..N."""
    small = np.bincount(labels.ravel()) < min_px
    small[0] = False
    labels = np.where(small[labels], 0, labels)
    return relabel_sequential(labels)[0]


def match_fields(pred: np.ndarray, gt: np.ndarray, iou_thresh: float = 0.5) -> dict:
    """
    Field-level (object) matching between two instance maps.
    With IoU > 0.5 each field can match at most one field on the other side.

    Returns counts (tp, fp, fn) plus the summed IoU of matched pairs, so they
    can be pooled over a whole validation set before computing F1.
    """
    pred, _, _ = relabel_sequential(pred.astype(np.int64))
    gt,   _, _ = relabel_sequential(gt.astype(np.int64))
    n_p, n_g = int(pred.max()), int(gt.max())
    if n_p == 0 or n_g == 0:
        return {'tp': 0, 'fp': n_p, 'fn': n_g, 'iou_sum': 0.0}
    ov = np.bincount(pred.ravel() * (n_g + 1) + gt.ravel(),
                     minlength=(n_p + 1) * (n_g + 1)).reshape(n_p + 1, n_g + 1)
    inter = ov[1:, 1:].astype(np.float64)
    union = ov[1:, :].sum(1)[:, None] + ov[:, 1:].sum(0)[None, :] - inter
    iou   = inter / np.maximum(union, 1)
    hit   = iou > iou_thresh
    tp    = int(hit.sum())
    return {'tp': tp, 'fp': n_p - tp, 'fn': n_g - tp, 'iou_sum': float(iou[hit].sum())}


def summarise_matches(m: dict) -> dict:
    tp, fp, fn = m['tp'], m['fp'], m['fn']
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    return {'field_f1': 2 * prec * rec / max(prec + rec, 1e-9),
            'field_precision': prec, 'field_recall': rec,
            'matched_iou': m['iou_sum'] / max(tp, 1)}


def prepare_config(config: dict, device=None) -> dict:
    """
    Fill in defaults and cap CPU usage.

    cpu_threads limits the threads PyTorch and OpenCV may use. On a GPU the
    network runs on the GPU, so a couple of CPU threads are enough for the
    rest; on a CPU-only machine all cores are used unless you set a lower
    number (fewer threads = lower CPU load, but slower).
    """
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cfg = {'batch_size': 4, 'tta': False, 'smooth_px': 6.0, 'save_visualization': True,
           **config}
    if cfg.get('cpu_threads') is None:
        cfg['cpu_threads'] = 2 if device.type == 'cuda' else (os.cpu_count() or 1)
    torch.set_num_threads(cfg['cpu_threads'])
    cv2.setNumThreads(cfg['cpu_threads'])
    return cfg


def load_rgb(image_path: str):
    """
    Read the first three bands as uint8 RGB, plus a validity mask.

    The mask comes from the raster's nodata value / alpha band / internal
    mask (GeoTIFF) or the alpha channel (PNG). Tiles with no valid pixels —
    e.g. the black margins around an orthomosaic — are skipped at inference.
    """
    try:
        with rasterio.open(image_path) as src:
            arr   = src.read([1, 2, 3]).transpose(1, 2, 0)
            valid = src.dataset_mask() > 0
        if arr.dtype != np.uint8:   # e.g. 16-bit orthomosaics: stretch 2–98 %
            sample = arr[::8, ::8][valid[::8, ::8]]          # subsample: fast on huge rasters
            lo, hi = np.percentile(sample, (2, 98)) if sample.size else (0, 1)
            arr = np.clip((arr - lo) * (255.0 / max(hi - lo, 1e-6)), 0, 255).astype(np.uint8)
        return np.ascontiguousarray(arr), valid
    except Exception:
        im = Image.open(image_path)
        valid = np.array(im.getchannel('A')) > 0 if 'A' in im.getbands() else None
        rgb = np.array(im.convert('RGB'))
        return rgb, (valid if valid is not None else np.ones(rgb.shape[:2], bool))


def _blend_window(size: int, floor: float = 0.05) -> np.ndarray:
    """
    2-D raised-cosine blending window with a small floor.
    A plain Hann window is exactly 0 on its outer rows/columns, which would
    zero the probabilities along the image border.
    """
    w1d = np.hanning(size).astype(np.float32)
    return floor + (1.0 - floor) * np.outer(w1d, w1d)


def _predict(model, batch, tta: bool):
    """
    Field / boundary / distance probabilities for a batch (on the device).
    TTA (off by default) averages identity + h-flip + v-flip: 3× the compute.
    """
    flips = [[], [-1], [-2]] if tta else [[]]
    acc = 0.0
    for dims in flips:
        out = model(torch.flip(batch, dims) if dims else batch)
        # P(field) for a 2-class softmax = sigmoid(logit_field − logit_background)
        p = torch.sigmoid(torch.cat([out['seg'][:, 1:2] - out['seg'][:, 0:1],
                                     out['bound'], out['dist']], 1).float())
        acc = acc + (torch.flip(p, dims) if dims else p)
    return acc / len(flips)                                    # (B, 3, H, W)


@torch.inference_mode()
def run_tiled_inference(img_np: np.ndarray, model, config: dict, valid: np.ndarray = None):
    """
    Tiled inference with blended overlaps.

    Speed / CPU measures:
      • tiles without any valid pixel (nodata) are skipped
      • uint8 tiles go to the device as-is; normalisation runs on the device
      • blending accumulates on the GPU (when it fits), so the CPU doesn't
        touch every pixel of every tile and never waits on the GPU per batch
      • TTA is off by default (set config['tta']=True for ~3× compute)

    The image is reflect-padded by half the overlap so border pixels never sit
    at the very edge of a tile. Returns float32 maps in [0,1]:
    field_p, bound_p, dist_p (0 on nodata).
    """
    tile, overlap = config['tile_size'], config['overlap']
    stride = tile - overlap
    pad    = overlap // 2
    H, W   = img_np.shape[:2]
    device = next(model.seg_head.parameters()).device

    ph = max(tile - (H + 2 * pad), 0)
    pw = max(tile - (W + 2 * pad), 0)
    padded = np.pad(img_np, ((pad, pad + ph), (pad, pad + pw), (0, 0)), mode='reflect')
    PH, PW = padded.shape[:2]
    if valid is None:
        valid = np.ones((H, W), bool)
    valid_p = np.pad(valid, ((pad, pad + ph), (pad, pad + pw)))

    ys = list(range(0, PH - tile + 1, stride))
    xs = list(range(0, PW - tile + 1, stride))
    if ys[-1] + tile < PH: ys.append(PH - tile)
    if xs[-1] + tile < PW: xs.append(PW - tile)
    # Integral image → O(1) "does this tile contain valid pixels?" check
    ii = np.pad(valid_p.astype(np.int64).cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    coords = [(y, x) for y in ys for x in xs
              if ii[y + tile, x + tile] - ii[y, x + tile] - ii[y + tile, x] + ii[y, x] > 0]
    n_all = len(ys) * len(xs)
    if len(coords) < n_all:
        print(f'  Skipping {n_all - len(coords)}/{n_all} tiles with no valid pixels')

    # Accumulate on the GPU if 4 float32 maps fit comfortably, else on CPU
    acc_dev = device
    if device.type == 'cuda':
        free, _ = torch.cuda.mem_get_info(device)
        if 4 * 4 * PH * PW > 0.5 * free:
            acc_dev = torch.device('cpu')
    acc = torch.zeros((4, PH, PW), dtype=torch.float32, device=acc_dev)   # field, bound, dist, weight
    win = torch.from_numpy(_blend_window(tile)).to(acc_dev)

    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1) * 255.0
    std  = torch.tensor(IMAGENET_STD,  device=device).view(1, 3, 1, 1) * 255.0
    cuda = device.type == 'cuda'
    bs   = config.get('batch_size', 4)

    for i in tqdm(range(0, len(coords), bs), desc='Tiled inference'):
        chunk = coords[i:i + bs]
        batch = torch.from_numpy(np.stack([padded[y:y + tile, x:x + tile] for y, x in chunk]))
        if cuda:
            batch = batch.pin_memory()
        batch = batch.to(device, non_blocking=True).permute(0, 3, 1, 2).float()
        batch = (batch - mean) / std
        if cuda:
            batch = batch.contiguous(memory_format=torch.channels_last)
        with autocast_ctx(device):
            probs = _predict(model, batch, config.get('tta', False)).to(acc_dev)
        for j, (y, x) in enumerate(chunk):
            acc[:3, y:y + tile, x:x + tile].addcmul_(probs[j], win)
            acc[3,  y:y + tile, x:x + tile].add_(win)

    crop = (slice(None), slice(pad, pad + H), slice(pad, pad + W))
    acc  = acc[crop]
    out  = (acc[:3] / acc[3:].clamp_min(1e-6)).cpu().numpy()
    out[:, ~valid] = 0.0
    del acc
    return out[0], out[1], out[2]


def labels_to_geodataframe(labels: np.ndarray, image_path: str, config: dict) -> gpd.GeoDataFrame:
    """
    Instance label map → one smooth, low-vertex polygon per field.

      1. rasterio.features.shapes traces the pixel edges (exact, but every
         pixel step on a slanted edge becomes two vertices)
      2. small background holes are filled; each field keeps its largest part
      3. all fields are simplified TOGETHER with shapely.coverage_simplify
         (Visvalingam–Whyatt on the shared-edge network): pixel staircases
         become straight edges, and each edge shared by two neighbours is
         simplified once, so neighbours stay gap- and overlap-free
         (per-polygon simplification made neighbours overlap)

    `smooth_px` sets how much detail goes: larger = fewer vertices and
    straighter edges. Coordinates are in the image CRS; non-georeferenced
    images stay in pixel coordinates with crs=None.
    """
    try:
        with rasterio.open(image_path) as src:
            transform, crs = src.transform, src.crs
    except Exception:
        transform, crs = rasterio.Affine.identity(), None
    georef = crs is not None and transform != rasterio.Affine.identity()
    if not georef:
        transform, crs = rasterio.Affine.identity(), None
        print('  ⚠ Image has no geotransform — polygons are in pixel coordinates')

    px_size  = abs(transform.a) if georef else 1.0
    min_hole = config.get('postprocess', POSTPROCESS_DEFAULTS)['min_field_px'] * px_size ** 2

    parts = {}
    for geom, value in rasterio.features.shapes(labels.astype(np.int32),
                                                mask=labels > 0, transform=transform):
        parts.setdefault(int(value), []).append(shape(geom))

    ids, geoms = [], []
    for field_id, polys in sorted(parts.items()):
        g = max(polys, key=lambda p: p.area)        # pieces of one field never overlap
        # Holes smaller than the minimum field size can't contain another field
        holes = [r for r in g.interiors if Polygon(r).area >= min_hole]
        ids.append(field_id)
        geoms.append(Polygon(g.exterior, holes))

    n_raw = int(sum(shapely.get_num_coordinates(geoms))) if geoms else 0
    tol   = config.get('smooth_px', 6.0) * px_size
    if geoms and tol > 0:
        try:
            geoms = list(shapely.coverage_simplify(np.array(geoms, dtype=object), tol))
        except Exception as e:   # shapely < 2.1 / GEOS < 3.12
            print(f'  ⚠ coverage_simplify unavailable ({type(e).__name__}); '
                  'simplifying fields one by one (neighbours may overlap slightly)')
            geoms = [g.simplify(tol, preserve_topology=True) for g in geoms]

    areas = np.bincount(labels.ravel())
    rows, keep = [], []
    for field_id, g in zip(ids, geoms):
        if not g.is_valid:
            g = shapely.make_valid(g)
            if isinstance(g, (MultiPolygon, shapely.GeometryCollection)):
                polys = [p for p in getattr(g, 'geoms', []) if isinstance(p, Polygon)]
                g = max(polys, key=lambda p: p.area) if polys else Polygon()
        if g.is_empty:
            continue
        rows.append({'field_id': field_id,
                     'area_px':  int(areas[field_id]),
                     'area_m2':  round(g.area, 2) if georef and crs.is_projected else None,
                     'n_vertices': int(shapely.get_num_coordinates(g))})
        keep.append(g)

    gdf = gpd.GeoDataFrame(rows, geometry=keep, crs=crs)
    if len(gdf):
        print(f'  GeoDataFrame: {len(gdf)} fields  |  CRS={gdf.crs}  |  vertices '
              f'{n_raw:,} → {int(gdf.n_vertices.sum()):,} '
              f'(mean {gdf.n_vertices.mean():.0f} per field, smooth_px={config.get("smooth_px", 6.0)})')
    return gdf


FILTER_DEFAULTS = {
    # ── Circular objects (no circular fields in the Philippine context) ──────
    # A field is removed only if it is round by BOTH measures, so compact
    # polygonal fields (pentagons, hexagons) and irregular fields are kept.
    'filter_circular'   : True,
    'circularity_min'   : 0.90,  # Polsby-Popper 4πA/P²: circle 1.0, octagon 0.95,
                                 # hexagon 0.91, square 0.785, 2:1 rectangle 0.70
    'circle_fill_min'   : 0.85,  # area / smallest enclosing circle: circle 1.0,
                                 # octagon 0.90, hexagon 0.83, square 0.64

    # ── Implausibly small fields (relative to the scene) ─────────────────────
    'filter_small'      : True,
    'rel_area_min'      : 0.05,  # remove if area < 5 % of the scene's MEDIAN field area…
    'min_fields_for_rel': 10,    # …only when there are enough fields for a reliable median
    'small_cap_m2'      : 200.0, # …and never remove a field larger than this (georeferenced only)
    'keep_edge_fields'  : True,  # never remove fields cut by the image edge / nodata
}


def edge_field_ids(labels: np.ndarray, valid: np.ndarray = None) -> set:
    """Fields touching the image border or a nodata area (often only partly visible)."""
    edge = np.zeros(labels.shape, bool)
    edge[0, :] = edge[-1, :] = edge[:, 0] = edge[:, -1] = True
    if valid is not None and not valid.all():
        edge |= cv2.dilate((~valid).astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return set(np.unique(labels[edge]).tolist()) - {0}


def shape_metrics(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Add circularity (Polsby-Popper, outer ring only) and circle_fill
    (area / area of the smallest enclosing circle). Both are computed on the
    smoothed polygons: on raw pixel outlines the staircase perimeter would
    make even a perfect circle look irregular.
    """
    outer = shapely.polygons(shapely.get_exterior_ring(gdf.geometry.values))
    area  = shapely.area(outer)
    perim = shapely.length(outer)
    r     = shapely.minimum_bounding_radius(outer)
    gdf = gdf.copy()
    gdf['circularity'] = np.round(4 * np.pi * area / np.maximum(perim, 1e-12) ** 2, 3)
    gdf['circle_fill'] = np.round(area / np.maximum(np.pi * r ** 2, 1e-12), 3)
    return gdf


def filter_fields(gdf: gpd.GeoDataFrame, labels: np.ndarray, valid: np.ndarray = None,
                  cfg: dict = FILTER_DEFAULTS):
    """
    Remove implausible fields. Returns (kept, removed); `removed` has a
    `filter_reason` column so every removal can be reviewed (it is saved to
    a separate GeoJSON by the pipeline, nothing is silently lost).
    """
    cfg = {**FILTER_DEFAULTS, **(cfg or {})}
    if len(gdf) == 0:
        return gdf, gdf.assign(filter_reason=[])

    gdf = shape_metrics(gdf)
    edge_ids = edge_field_ids(labels, valid)
    gdf['touches_edge'] = gdf.field_id.isin(edge_ids)
    area = gdf.geometry.area
    med  = float(np.median(area))
    gdf['area_rel'] = np.round(area / med, 3) if med > 0 else np.nan

    reason = pd.Series('', index=gdf.index, dtype=object)

    if cfg['filter_circular']:
        circ = (gdf.circularity >= cfg['circularity_min']) & (gdf.circle_fill >= cfg['circle_fill_min'])
        reason[circ] = 'circular'

    if cfg['filter_small'] and len(gdf) >= cfg['min_fields_for_rel']:
        small = area < cfg['rel_area_min'] * med
        projected = gdf.crs is not None and gdf.crs.is_projected
        if projected and cfg.get('small_cap_m2') is not None:
            small &= area < cfg['small_cap_m2']
        if cfg['keep_edge_fields']:
            small &= ~gdf.touches_edge
        reason[small & (reason == '')] = 'too_small'

    removed = gdf[reason != ''].assign(filter_reason=reason[reason != ''])
    kept    = gdf[reason == '']
    counts  = removed.filter_reason.value_counts().to_dict()
    print(f'  Filters: kept {len(kept)} / {len(gdf)} fields'
          + (f'  |  removed {counts}' if counts else '')
          + f'  |  median area {med:,.1f} {"m²" if gdf.crs is not None and gdf.crs.is_projected else "units²"}')
    return kept, removed

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Inference configuration
#
# The watershed thresholds come from the checkpoint (tuned on the validation
# set by the training notebook). Override any of them here if needed.
# ════════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Tiling ────────────────────────────────────────────────────────────
    'tile_size'      : MODEL_CONFIG.get('img_size', 512),   # = training crop size
    'overlap'        : 128,
    'batch_size'     : 4,      # tiles per forward pass (raise on big GPUs)
    'tta'            : False,  # True = average over h/v flips (3× slower, slightly smoother maps)

    # ── CPU usage ─────────────────────────────────────────────────────────
    'cpu_threads'    : None,   # None = 2 on GPU runtimes, all cores on CPU-only; set e.g. 2 to cap

    # ── Field delineation (seeded watershed) ──────────────────────────────
    'postprocess'    : {**POSTPROCESS_DEFAULTS, **ckpt.get('postprocess', {})},

    # ── Polygon export ────────────────────────────────────────────────────
    'smooth_px'      : 6.0,    # simplification strength in pixels:
                               # higher = smoother edges, fewer vertices; lower = more detail

    # ── Field filters (removed fields are saved separately for review) ────
    # Defaults: drop near-perfect circles; drop fields < 5 % of the median
    # field area, but only with ≥ 10 fields, below 200 m², not at the edge.
    'filters'        : {**FILTER_DEFAULTS},

    # ── Output ────────────────────────────────────────────────────────────
    'save_visualization': True,
    'output_dir'     : OUTPUT_DIR,
    'image_name'     : 'cropfield_dino_fractal',
}
CONFIG = prepare_config(CONFIG, DEVICE)
print('Post-processing:', CONFIG['postprocess'])
print(f"CPU threads: {CONFIG['cpu_threads']}  |  TTA: {CONFIG['tta']}  |  smooth_px: {CONFIG['smooth_px']}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Visualization
# ════════════════════════════════════════════════════════════════════════════

def visualise_results(img_np, field_p, bound_p, labels, removed_ids=(), save_path=None, max_side=2048):
    """
    Original | field probability | boundary probability | delineated fields.
    Fields removed by the filters are outlined in red.
    Large images are downsampled for display (max_side px), which keeps
    plotting fast and light on memory for big orthomosaics.
    """
    step = max(1, int(np.ceil(max(labels.shape) / max_side)))
    if step > 1:
        img_np, field_p, bound_p, labels = (a[::step, ::step] for a in (img_np, field_p, bound_p, labels))
    rng = np.random.default_rng(0)
    lut = rng.integers(40, 255, (labels.max() + 1, 3), dtype=np.uint8); lut[0] = 0
    is_removed = np.zeros(labels.max() + 1, bool)
    is_removed[[i for i in removed_ids if i <= labels.max()]] = True
    overlay = img_np.copy()
    fields  = (labels > 0) & ~is_removed[labels]
    overlay[fields] = (0.55 * img_np[fields] + 0.45 * lut[labels][fields]).astype(np.uint8)
    edges = find_edges(labels)
    overlay[edges & ~is_removed[labels]] = (255, 165, 0)
    overlay[edges &  is_removed[labels]] = (255, 0, 0)

    fig, axes = plt.subplots(1, 4, figsize=(26, 6))
    axes[0].imshow(img_np);                           axes[0].set_title('Original')
    axes[1].imshow(field_p, cmap='magma', vmin=0, vmax=1); axes[1].set_title('Field probability')
    axes[2].imshow(bound_p, cmap='hot',   vmin=0, vmax=1); axes[2].set_title('Boundary probability')
    axes[3].imshow(overlay);                          axes[3].set_title(f'Fields: {labels.max() - is_removed.sum()} kept (orange), '
                                                                f'{is_removed.sum()} removed (red)')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'  Saved → {save_path}')
    plt.show()


def find_edges(labels):
    """1-px outlines of every field (edges between fields included)."""
    e = np.zeros(labels.shape, bool)
    e[:-1] |= labels[:-1] != labels[1:]
    e[:, :-1] |= labels[:, :-1] != labels[:, 1:]
    return e & (labels > 0)


def print_field_stats(gdf):
    print(f'\n  ── Field statistics ──────────────────────────')
    print(f'  Count          : {len(gdf)}')
    if len(gdf):
        print(f'  Area px  min   : {gdf.area_px.min():,.0f}')
        print(f'  Area px  mean  : {gdf.area_px.mean():,.0f}')
        print(f'  Area px  max   : {gdf.area_px.max():,.0f}')
        if gdf.area_m2.notna().any():
            print(f'  Area m²  mean  : {gdf.area_m2.mean():,.1f}')
        print(f'  Vertices mean  : {gdf.n_vertices.mean():,.1f}  per polygon')
    print(f'  ──────────────────────────────────────────────')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Inference Pipeline
# ════════════════════════════════════════════════════════════════════════════

def run_inference(image_path: str, model, config: dict):
    """
    Pipeline:
      1. Tiled forward pass (nodata tiles skipped) → field / boundary / distance maps
      2. Seeded watershed → one label per field (touching fields separated)
      3. Smooth low-vertex polygons (shared edges simplified together),
         georeferenced when the image has a CRS
      4. Filters: circular objects and implausibly small fields
      5. GeoJSON export (kept + removed-for-review) + visualisation
    """
    t0 = time.time()
    print(f"\n{'='*62}\n  Inference Pipeline\n{'='*62}")
    print(f'  Image : {image_path}')

    print('\n[1/5] Tiled inference…')
    img_np, valid = load_rgb(image_path)
    print(f'  Image size: {img_np.shape[1]}×{img_np.shape[0]}  |  valid {100 * valid.mean():.0f}%')
    field_p, bound_p, dist_p = run_tiled_inference(img_np, model, config, valid=valid)

    print('\n[2/5] Field delineation (watershed)…')
    labels = delineate_fields(field_p, bound_p, dist_p, config['postprocess'])
    print(f'  {labels.max()} fields  |  field coverage {100 * (labels > 0).mean():.1f}%  |  '
          f'boundary coverage {100 * (bound_p > 0.5).mean():.1f}%')
    if labels.max() == 0:
        print('  ⚠ No fields detected. Try lowering extent_thresh / marker_dist.')
        return None, None, labels

    print('\n[3/5] Polygons + georeferencing…')
    gdf = labels_to_geodataframe(labels, image_path, config)

    print('\n[4/5] Field filters…')
    gdf, removed = filter_fields(gdf, labels, valid, config.get('filters'))

    print('\n[5/5] Export…')
    name    = config['image_name']
    geojson = os.path.join(config['output_dir'], f'{name}_fields.geojson')
    rem_js  = os.path.join(config['output_dir'], f'{name}_removed_fields.geojson')
    vis_png = os.path.join(config['output_dir'], f'{name}_detections.png')
    gdf.to_file(geojson, driver='GeoJSON')
    print(f'  GeoJSON → {geojson}')
    if len(removed):
        removed.to_file(rem_js, driver='GeoJSON')
        print(f'  Removed fields (for review) → {rem_js}')
    elif os.path.exists(rem_js):
        os.remove(rem_js)   # don't leave a stale file from an earlier run

    if config.get('save_visualization', True):
        visualise_results(img_np, field_p, bound_p, labels, removed_ids=removed.field_id.tolist(),
                          save_path=vis_png)
    print_field_stats(gdf)
    del field_p, bound_p, dist_p
    gc.collect()
    print(f"\n{'='*62}\n  ✓ Done in {time.time() - t0:.1f}s  |  {len(gdf)} fields\n{'='*62}\n")
    return gdf, removed, labels

In [ ]:
IMAGE_PATH = '/content/drive/MyDrive/AGRI/CropField_Segmentation/Screenshot 2026-04-20 203251.png'

gdf, removed, labels = run_inference(IMAGE_PATH, model, CONFIG)